# Свёрточная нейронная сеть: распознавание изображений

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Свёрточная нейронная сеть».

## Что делает метод

Свёрточная нейронная сеть учится «видеть» закономерности на изображениях. Исследователь показывает ей размеченные снимки (например, клетки разных типов или кадры эксперимента), а сеть учится распознавать такие же объекты на новых изображениях: относить снимок к классу, обводить области, считать объекты.

Сеть применяет обучаемые фильтры (свёртки), выделяя сначала простые детали — края и пятна, — а затем всё более сложные образы. В этом блокноте мы обучим компактную свёрточную сеть классифицировать изображения и разберём, как читать результат. Блокнот рассчитан на запуск без графического ускорителя; время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы рассматриваете микрофотографию клетки через лупу, перемещая её по снимку. В каждом положении вы замечаете мелкие детали: острый край мембраны, тёмное пятно ядра, характерный узор органелл. Потом вы мысленно «складываете» все замеченные детали вместе и делаете вывод: «это клетка данного типа».

Свёрточная нейронная сеть делает то же самое, только автоматически. Небольшие **фильтры** (свёртки) — аналог вашей лупы — скользят по изображению и реагируют на конкретные детали: края, пятна, градиенты яркости. Первые слои сети улавливают простые детали, последующие — всё более сложные образы, собранные из этих деталей. Финальный слой принимает решение о классе объекта, опираясь на набор найденных признаков.

**Ключевые термины, которые встретятся в блокноте:**

- **Нейрон** — вычислительная единица, получающая несколько входов, взвешивающая их и передающая один выход.
- **Слой** — группа нейронов, выполняющих одну операцию над всем входным тензором.
- **Свёрточный слой** — слой, в котором каждый нейрон смотрит лишь на небольшое окно входного изображения; одни и те же веса применяются ко всем позициям (это и есть «свёртка»).
- **Функция потерь (loss)** — число, показывающее, насколько предсказания сети расходятся с правильными ответами; задача обучения — минимизировать её.
- **Эпоха** — один полный проход обучающего набора через сеть; обычно нужно несколько десятков эпох, чтобы сеть сошлась.
- **Градиент** — направление наибыстрейшего роста функции потерь; сеть движется в противоположном направлении, постепенно улучшая предсказания.

## 1. Установка библиотек

В Google Colab `torch` уже установлен. Ячейка ниже гарантирует наличие нужных пакетов.

In [ ]:
%pip install -q torch scikit-learn numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор рукописных цифр (`digits` из `scikit-learn`): 1797 изображений размером 8×8 в оттенках серого, десять классов. Малый размер снимков позволяет обучить сеть за минуту без графического ускорителя, сохраняя суть метода. Для реальных научных снимков подход тот же — меняются лишь размер изображений и архитектура.

In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

data = load_digits()
# Приводим к формату изображений: (наблюдение, канал, высота, ширина).
images = data.images.astype("float32") / 16.0          # нормировка яркости в [0, 1]
X = images[:, None, :, :]
y = data.target.astype("int64")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Обучающих снимков: {len(X_train)}, тестовых: {len(X_test)}")
print(f"Размер одного снимка: {X_train.shape[1:]}")

### Как выглядят данные

Прежде чем обучать сеть, полезно взглянуть на несколько образцов. Ячейка ниже показывает двадцать снимков из обучающего набора с указанием истинного класса (цифры). Это помогает убедиться, что данные загружены корректно, и оценить сложность задачи «на глаз».

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for ax, idx in zip(axes.ravel(), range(20)):
    ax.imshow(X_train[idx, 0], cmap="gray_r")
    ax.set_title(str(y_train[idx]), fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle("Примеры обучающих снимков (цифра — истинный класс)", fontsize=13)
fig.tight_layout()
plt.show()

## 4. Применение метода

### Шаг 1. Архитектура сети

Соберём компактную свёрточную сеть. Она состоит из двух частей:

1. **Извлечение признаков** — два свёрточных блока. Каждый блок: свёрточный слой (фильтры скользят по изображению и находят характерные детали) → нелинейная активация ReLU (обнуляет отрицательные значения, придавая сети способность учить сложные зависимости) → слой объединения MaxPool (берёт максимум в окне 2×2, уменьшая карту вдвое и делая признаки устойчивее к сдвигу).
2. **Классификатор** — полносвязные слои (каждый нейрон связан со всеми предыдущими), которые на основе найденных признаков принимают решение о классе. Слой Dropout случайно «выключает» нейроны во время обучения, не давая сети переобучиться.

Конфигурация специально упрощена для быстрого запуска без GPU.

In [ ]:
import torch.nn as nn


class SmallCNN(nn.Module):
    """Компактная свёрточная сеть для классификации изображений 8x8."""

    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # свёртка: 16 фильтров
            nn.ReLU(),
            nn.MaxPool2d(2),                              # объединение 8x8 -> 4x4
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # свёртка: 32 фильтра
            nn.ReLU(),
            nn.MaxPool2d(2),                              # объединение 4x4 -> 2x2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallCNN().to(device)
print(f"Устройство вычислений: {device}")
print(model)

### Шаг 2. Данные, цикл обучения и оценка качества

Ячейка ниже:
- Упаковывает массивы в `DataLoader` — итератор, подающий данные **батчами** (пакетами по 64 снимка). Это эффективнее, чем обновлять веса по одному снимку.
- Задаёт **функцию потерь** `CrossEntropyLoss` — стандартный выбор для задачи классификации на несколько классов.
- Задаёт **оптимизатор** `Adam` — алгоритм градиентного спуска с адаптивным шагом обучения.
- Запускает цикл из 25 эпох. В каждой эпохе: прямой проход (предсказание) → подсчёт потерь → обратный проход (`loss.backward()`, вычисление градиентов) → шаг оптимизатора (обновление весов).

После каждой эпохи оцениваем качество на тестовом наборе — данных, которые сеть не видела во время обучения.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def evaluate(loader):
    """Возвращает среднюю потерю и долю верных ответов на наборе."""
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss_sum += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / total, correct / total


n_epochs = 25
history = {"train_loss": [], "test_loss": [], "test_acc": []}
for epoch in range(1, n_epochs + 1):
    model.train()
    epoch_loss, seen = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(yb)
        seen += len(yb)
    test_loss, test_acc = evaluate(test_loader)
    history["train_loss"].append(epoch_loss / seen)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Эпоха {epoch:2d}: потеря на тесте {test_loss:.4f}, "
              f"доля верных {test_acc:.3f}")

### Шаг 3. Кривые обучения, матрица ошибок и примеры предсказаний

Три визуализации ниже дают три разных взгляда на качество обученной модели:
1. **Кривые потерь** — динамика улучшения по эпохам.
2. **Матрица ошибок** — какие классы путает модель.
3. **Примеры предсказаний** — наглядная проверка на конкретных снимках.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Предсказания на всём тесте.
model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_test).to(device))
    y_pred = logits.argmax(1).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
epochs = range(1, n_epochs + 1)
axes[0].plot(epochs, history["train_loss"], color=VIZ["series"][0],
             label="Обучение")
axes[0].plot(epochs, history["test_loss"], color=VIZ["series"][2],
             label="Проверка")
axes[0].set_title("Кривые обучения")
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Функция потерь")
axes[0].legend(loc="upper right")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, cmap="Blues", ax=axes[1], colorbar=False)
axes[1].set_title("Матрица ошибок")
axes[1].set_xlabel("Предсказанный класс")
axes[1].set_ylabel("Истинный класс")
axes[1].grid(False)

fig.tight_layout()
plt.show()

### Как читать кривые обучения и матрицу ошибок

**Кривые обучения (левый график):**
- Обе линии должны снижаться — это означает, что сеть учится.
- Если потеря на обучении (синяя) продолжает падать, а на проверке (оранжевая) начинает расти — это **переобучение**: сеть запомнила обучающий набор, но не обобщила знания. Помогают: слой Dropout, аугментация данных, меньшая сеть, ранняя остановка.
- Если обе линии «плато» на высоком уровне — сеть **недообучилась**: нужно больше эпох, другой шаг обучения или более широкая архитектура.

**Матрица ошибок (правый график):**
- По горизонтали — что **предсказала** сеть, по вертикали — что **на самом деле** на снимке.
- Числа на **главной диагонали** — верные ответы; чем они больше, тем лучше.
- Числа **вне диагонали** — ошибки. Скопление ошибок на пересечении двух классов указывает, что именно эти категории сеть путает (обычно визуально похожие).

In [ ]:
# Примеры тестовых снимков с предсказаниями сети.
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for ax, idx in zip(axes.ravel(), range(12)):
    ax.imshow(X_test[idx, 0], cmap="gray_r")
    correct = y_pred[idx] == y_test[idx]
    color = VIZ["series"][1] if correct else VIZ["series"][2]
    ax.set_title(f"Прогноз: {y_pred[idx]}", color=color, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle("Примеры предсказаний (зелёный — верно, оранжевый — ошибка)",
             fontsize=13)
fig.tight_layout()
plt.show()

### Как читать примеры предсказаний

Заголовок каждого снимка — это предсказание сети. **Зелёный** цвет заголовка означает верный ответ, **оранжевый** — ошибку. Обратите внимание: ошибки, как правило, приходятся на нетипичные или небрежно написанные цифры — именно те случаи, где у человека тоже возникли бы сомнения.

### «Ага»-эксперимент: что видит каждый фильтр первого свёрточного слоя

Главная идея свёрточной сети — **иерархия признаков**: первый слой реагирует на простые детали (края, штрихи), второй — на комбинации этих деталей, и так далее. Ниже мы возьмём один тестовый снимок и посмотрим, как 16 фильтров первого слоя «видят» его. Каждая мини-карта показывает, на что именно реагирует один фильтр: яркие пиксели означают высокую активацию (фильтр «нашёл» свой признак), тёмные — низкую.

In [ ]:
# Извлекаем карты активаций первого свёрточного слоя для одного снимка.
sample_idx = 0
sample_img = torch.from_numpy(X_test[sample_idx : sample_idx + 1]).to(device)
true_label = y_test[sample_idx]
pred_label = y_pred[sample_idx]

model.eval()
with torch.no_grad():
    # Пропускаем через первый Conv2d + ReLU вручную, чтобы получить карты.
    conv1 = model.features[0]   # первый свёрточный слой
    relu1 = model.features[1]   # ReLU
    feat_maps = relu1(conv1(sample_img)).cpu().numpy()[0]  # (16, 8, 8)

n_filters = feat_maps.shape[0]
fig, axes = plt.subplots(3, 6, figsize=(13, 7))
axes = axes.ravel()

# Первая ячейка — исходный снимок.
axes[0].imshow(X_test[sample_idx, 0], cmap="gray_r")
axes[0].set_title(f"Исходник\nИстина: {true_label}, прогноз: {pred_label}",
                  fontsize=10)
axes[0].set_xticks([])
axes[0].set_yticks([])
axes[0].grid(False)

# Остальные 16 ячеек — карты признаков.
for i in range(n_filters):
    ax = axes[i + 1]
    ax.imshow(feat_maps[i], cmap="Blues")
    ax.set_title(f"Фильтр {i + 1}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)

# Последняя ячейка — пустая.
axes[-1].set_visible(False)

fig.suptitle("Карты активаций первого свёрточного слоя\n"
             "(каждая ячейка — «взгляд» одного фильтра на снимок)",
             fontsize=13)
fig.tight_layout()
plt.show()

### Как читать карты активаций

Каждая мини-карта — это ответ одного фильтра на исходный снимок. Яркие (насыщенные) пиксели означают, что фильтр «нашёл» свой признак в данном месте изображения; тёмные — что ничего характерного там нет.

Посмотрите: одни фильтры светлеют на вертикальных штрихах цифры, другие — на горизонтальных или диагональных, третьи реагируют на углы. Именно из таких элементарных откликов следующий свёрточный слой собирает более сложные составные признаки. На реальных научных снимках высокого разрешения этот эффект выражен ещё ярче: первые слои выделяют края клеток или зёрна, поздние — целые структуры.

## 5. Интерпретация результата

- **Кривые обучения**: обе линии должны снижаться. Если потеря на проверке начинает расти, а на обучении продолжает падать — это переобучение; помогают аугментация данных, регуляризация (`Dropout`) и ранняя остановка.
- **Доля верных ответов** на тесте — общая точность. Для несбалансированных классов смотрите также метрики по каждому классу (строки матрицы ошибок).
- **Матрица ошибок**: на диагонали — верные ответы; сосредоточение ошибок в одной паре классов указывает на визуально похожие категории.
- **Карты активаций** показывают, что именно «видит» сеть: первые слои реагируют на штрихи и края, поздние — на составные структуры. Это помогает отлавливать ситуации, когда сеть опирается на артефакты (освещение, фон), а не на объект.

Помните: для научных задач решающее значение имеет проверка на снимках из **независимого эксперимента** — различия в подготовке препаратов и освещении сеть может ошибочно принять за полезный сигнал.

## 6. Попробуйте на своих данных

Для своих изображений подготовьте массив `X` формы `(наблюдение, канал, высота, ширина)` и массив меток `y`. Снимки приводят к единому размеру и нормируют яркость в диапазон [0, 1].

1. Загрузите изображения в Colab (папкой или архивом) через панель «Файлы».
2. Снимите комментарии в ячейке ниже и при необходимости измените размер входа и число классов в `SmallCNN`.
3. Для снимков заметно крупнее 8×8 практичнее перенос обучения — дообучение готовой сети (например, ResNet из `torchvision`) вместо обучения с нуля.
4. Последовательно выполните ячейки разделов 4 и 5.

In [ ]:
# --- Шаблон загрузки своих изображений ---
# from pathlib import Path
# from PIL import Image
#
# image_size = 8                                # сторона снимка после приведения
# root = Path("my_images")                      # папка: подпапка на каждый класс
# classes = sorted(p.name for p in root.iterdir() if p.is_dir())
#
# arrays, labels = [], []
# for cls_idx, cls in enumerate(classes):
#     for path in (root / cls).glob("*"):
#         img = Image.open(path).convert("L").resize((image_size, image_size))
#         arrays.append(np.asarray(img, dtype="float32") / 255.0)
#         labels.append(cls_idx)
# X = np.stack(arrays)[:, None, :, :]
# y = np.array(labels, dtype="int64")
#
# print(f"Снимков: {len(X)}, классов: {len(classes)}")
# Далее повторите ячейки раздела 4.

## 7. Поэкспериментируйте сами

Ниже — конкретные параметры, которые стоит поменять, чтобы увидеть, как меняется поведение модели. Каждый эксперимент рассчитан на 1–2 минуты перезапуска.

**Эксперимент 1. Переобучение.**
В определении `SmallCNN` удалите строку `nn.Dropout(0.3)`, затем переобучите модель. Проследите на кривых обучения, как потеря на проверке начинает расти, пока потеря на обучении продолжает снижаться.

**Эксперимент 2. Число фильтров.**
Замените `nn.Conv2d(1, 16, ...)` на `nn.Conv2d(1, 4, ...)` (меньше фильтров) и снова обучите. Как изменились кривые и точность? Что видно на картах активаций?

**Эксперимент 3. Темп обучения.**
В строке `optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)` попробуйте `lr=1e-1` (слишком большой) и `lr=1e-5` (слишком маленький). Посмотрите на кривые: что происходит в каждом случае?

**Эксперимент 4. Другой снимок на картах активаций.**
В ячейке с картами активаций измените `sample_idx = 0` на любое другое число от 0 до длины теста. Посмотрите, как карты меняются для разных цифр: какие фильтры оказываются «универсальными», а какие специфичны для конкретного класса?

## Краткие выводы

- Свёрточная нейронная сеть автоматически учится иерархии признаков: от краёв и штрихов в первых слоях до сложных структур в последних.
- Для научных задач с ограниченным числом снимков предпочтительнее **перенос обучения** (дообучение готовой сети), а не обучение с нуля.
- Обязательно разделяйте данные на обучающий и тестовый наборы — и проверяйте на снимках из независимого эксперимента.
- Кривые обучения и матрица ошибок — основные диагностические инструменты; карты активаций помогают понять, что именно «видит» сеть и нет ли артефактов.
- Для несбалансированных классов (например, редкие типы клеток) смотрите не только на общую точность, но и на метрики по каждому классу.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На кривых обучения потеря на обучающем наборе монотонно убывает до 0.05, а потеря на проверочном — снизилась до 0.18 и затем начала расти. Что происходит и какие три конкретных изменения архитектуры или процедуры обучения вы рассмотрите в первую очередь?
2. Карты активаций первого свёрточного слоя для цифры «1» и цифры «8» выглядят почти одинаково у половины фильтров. Что это говорит об этих фильтрах, и является ли это признаком проблемы?
3. Вы хотите применить обученную сеть к микрофотографиям клеток из другого прибора. Точность резко падает. Назовите наиболее вероятные причины и объясните, почему обучение с нуля может быть хуже, чем дообучение готовой сети.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Это классическое переобучение: сеть запомнила обучающую выборку, но не обобщила знания. Первые шаги: (а) увеличить коэффициент Dropout или добавить слой; (б) применить аугментацию данных (повороты, сдвиги, изменение яркости); (в) реализовать раннюю остановку — сохранять веса в момент минимальной потери на проверочном наборе.
2. Фильтры, дающие похожие карты для разных цифр, реагируют на низкоуровневые общие признаки — горизонтальные или вертикальные штрихи, края, — которые присутствуют в любой цифре. Это нормально и ожидаемо: первый слой универсален, специализация по классам происходит в более глубоких слоях. Проблемой было бы, если бы все фильтры были идентичны (коллапс признаков).
3. Вероятные причины: различия в окрашивании препаратов, освещении, увеличении или разрешении — сеть уловила артефакты конкретного прибора как полезный сигнал. Дообучение (transfer learning) предпочтительнее обучения с нуля при ограниченных данных: нижние слои уже содержат универсальные детекторы краёв и текстур, которые работают для любых изображений; переобучать нужно лишь верхние слои под специфику новых снимков.
</details>
